# 12 - AG vs KMeans cluster comparison

This notebook compares the genetic algorithm clustering against the KMeans baseline using geometry, biological enrichment, purity/entropy, and the KMeans-to-AG overlap matrix.


In [1]:
# =============================================================================
# CELL 1 - Imports and project root
# =============================================================================

from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Markdown, display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pago_pipeline.cluster_comparison import (
    DEFAULT_OUTPUT_ROOT,
    DEFAULT_PC_BIOLOGY_LATEST_DIRECTORY,
    ENRICHMENT_COMPARISON_FILE_NAME,
    GEOMETRIC_METRICS_FILE_NAME,
    KMEANS_GA_MATRIX_FILE_NAME,
    KMEANS_SUBDIVISION_FILE_NAME,
    PLOT_HTML_FILE_NAME,
    PURITY_COMPARISON_FILE_NAME,
    SUMMARY_FILE_NAME,
    create_cluster_comparison_snapshot,
)

print(f"Project root: {PROJECT_ROOT}")


Project root: C:\Programming\Python\pAgo-project


In [2]:
# =============================================================================
# CELL 2 - Configuration
# =============================================================================

PC_BIOLOGY_LATEST_DIRECTORY = DEFAULT_PC_BIOLOGY_LATEST_DIRECTORY
OUTPUT_ROOT_DIRECTORY = DEFAULT_OUTPUT_ROOT

# Biological categories below this count are grouped as other_rare for purity
# and entropy. Cluster labels are never collapsed.
MINIMUM_BIOLOGICAL_CATEGORY_COUNT = 5

# Minimum AG subcluster size used when deciding whether a KMeans cluster was
# meaningfully subdivided.
MINIMUM_GA_SUBCLUSTER_SIZE = 5

print(f"PC biology latest: {(PROJECT_ROOT / PC_BIOLOGY_LATEST_DIRECTORY).resolve()}")
print(f"Output root: {(PROJECT_ROOT / OUTPUT_ROOT_DIRECTORY).resolve()}")


PC biology latest: C:\Programming\Python\pAgo-project\data\04-analysis\pc_biology_interpretation\latest
Output root: C:\Programming\Python\pAgo-project\data\04-analysis\cluster_comparison


## Analysis design

The comparison produces:

- geometric clustering metrics for AG and KMeans;
- ARI/NMI agreement between AG and KMeans;
- Cramer's V side-by-side for biological enrichment;
- weighted majority purity and entropy by biological variable;
- the KMeans x AG overlap matrix;
- a summary of KMeans clusters subdivided by AG.

The analysis preserves cluster labels even when clusters are small. Rare-category collapsing is applied only to biological variables for homogeneity metrics.


In [3]:
# =============================================================================
# CELL 3 - Build reproducible comparison snapshot
# =============================================================================

result = create_cluster_comparison_snapshot(
    pc_biology_latest_directory=PROJECT_ROOT / PC_BIOLOGY_LATEST_DIRECTORY,
    output_root_directory=PROJECT_ROOT / OUTPUT_ROOT_DIRECTORY,
    minimum_biological_category_count=MINIMUM_BIOLOGICAL_CATEGORY_COUNT,
    minimum_ga_subcluster_size=MINIMUM_GA_SUBCLUSTER_SIZE,
    update_latest_directory=True,
    verbose=True,
)

print(f"Immutable snapshot: {result.snapshot_directory}")
print(f"Latest snapshot: {result.latest_directory}")


Loading PC biology interpretation artifacts...
Computing AG vs KMeans geometric metrics...
Computing purity, entropy, and KMeans-to-AG subdivisions...
Rendering comparison Markdown and HTML report...
Immutable snapshot: C:\Programming\Python\pAgo-project\data\04-analysis\cluster_comparison\snapshots\2026-06-14T20-52-14Z__q_d0f23edc196d
Latest snapshot: C:\Programming\Python\pAgo-project\data\04-analysis\cluster_comparison\latest


In [4]:
# =============================================================================
# CELL 4 - Load generated comparison artifacts
# =============================================================================

latest_directory = result.latest_directory

geometric_metrics = pd.read_csv(latest_directory / GEOMETRIC_METRICS_FILE_NAME)
enrichment_comparison = pd.read_csv(latest_directory / ENRICHMENT_COMPARISON_FILE_NAME)
purity_comparison = pd.read_csv(latest_directory / PURITY_COMPARISON_FILE_NAME)
kmeans_ga_matrix = pd.read_csv(latest_directory / KMEANS_GA_MATRIX_FILE_NAME)
kmeans_subdivision = pd.read_csv(latest_directory / KMEANS_SUBDIVISION_FILE_NAME)

print(f"Geometric metric rows: {len(geometric_metrics):,}")
print(f"Enrichment comparison rows: {len(enrichment_comparison):,}")
print(f"Purity/entropy comparison rows: {len(purity_comparison):,}")
print(f"KMeans x AG matrix shape: {kmeans_ga_matrix.shape}")

display(geometric_metrics)


Geometric metric rows: 3
Enrichment comparison rows: 12
Purity/entropy comparison rows: 12
KMeans x AG matrix shape: (10, 13)


,cluster_column,sample_count,cluster_count,minimum_cluster_size,maximum_cluster_size,largest_cluster_fraction,silhouette,davies_bouldin,calinski_harabasz,adjusted_rand_index,normalized_mutual_information
0,ga_cluster_label,1010,12.0,13.0,402.0,0.39802,0.419374,0.651399,415.978918,NaN,NaN
1,kmeans_cluster_label,1010,10.0,13.0,403.0,0.39901,0.390364,0.808067,289.737134,NaN,NaN
2,ag_vs_kmeans,1010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.929675,0.939023


In [5]:
# =============================================================================
# CELL 5 - AG vs KMeans biological enrichment and homogeneity
# =============================================================================

display(
    enrichment_comparison[
        [
            "biological_variable",
            "cramers_v_ag",
            "cramers_v_kmeans",
            "delta_cramers_v_ag_minus_kmeans",
            "better_by_cramers_v",
            "q_value_bh_ag",
            "q_value_bh_kmeans",
        ]
    ]
)

display(
    purity_comparison[
        [
            "biological_variable",
            "weighted_majority_purity_ag",
            "weighted_majority_purity_kmeans",
            "delta_weighted_majority_purity_ag_minus_kmeans",
            "cluster_size_weighted_normalized_entropy_ag",
            "cluster_size_weighted_normalized_entropy_kmeans",
            "delta_normalized_entropy_kmeans_minus_ag",
            "better_by_weighted_purity",
            "better_by_weighted_entropy",
        ]
    ]
)


,biological_variable,cramers_v_ag,cramers_v_kmeans,delta_cramers_v_ag_minus_kmeans,better_by_cramers_v,q_value_bh_ag,q_value_bh_kmeans
0,paper_has_piwi_catalytic_tetrad,0.458080,0.381533,0.076546,AG,3.114469e-39,4.375076e-27
1,paper_paz_type,0.591582,0.552027,0.039555,AG,6.408273e-135,8.134476e-119
2,paper_ago_type_family,0.501144,0.470378,0.030766,AG,3.030800e-138,1.187334e-123
3,paper_ago_type_raw,0.363685,0.338331,0.025353,AG,1.570218e-126,1.189894e-111
4,paper_mid_5oh_type,0.189057,0.170307,0.018750,AG,1.861438e-04,6.304508e-04
5,qc__primary_label,0.146575,0.141606,0.004969,AG,2.681106e-02,1.712896e-02
6,qc__qc_decision,0.476380,0.471609,0.004771,AG,4.948477e-83,7.357447e-84
7,taxonomy__04,0.468987,0.465232,0.003756,AG,0.000000e+00,1.104586e-296
8,paper_domain_architecture,0.469597,0.465874,0.003723,AG,2.156620e-80,1.218521e-81
9,paper_mid_5p_type,0.512809,0.510388,0.002420,AG,1.478243e-288,7.079087e-294


,biological_variable,weighted_majority_purity_ag,weighted_majority_purity_kmeans,delta_weighted_majority_purity_ag_minus_kmeans,cluster_size_weighted_normalized_entropy_ag,cluster_size_weighted_normalized_entropy_kmeans,delta_normalized_entropy_kmeans_minus_ag,better_by_weighted_purity,better_by_weighted_entropy
0,taxonomy__04,0.409505,0.387260,0.022245,0.580414,0.599557,0.019143,AG,AG
1,taxonomy__03,0.634366,0.614386,0.019980,0.437084,0.452888,0.015804,AG,AG
2,paper_ago_type_family,0.733663,0.716832,0.016832,0.426553,0.448635,0.022083,AG,AG
3,paper_ago_type_raw,0.713861,0.697030,0.016832,0.372842,0.390380,0.017538,AG,AG
4,paper_has_piwi_catalytic_tetrad,0.840594,0.824752,0.015842,0.498188,0.534707,0.036519,AG,AG
5,paper_paz_type,0.726733,0.712871,0.013861,0.536675,0.563556,0.026881,AG,AG
6,paper_mid_5oh_type,0.991089,0.991089,0.000000,0.052625,0.054767,0.002143,tie,AG
7,qc__primary_label,0.963366,0.963366,0.000000,0.203414,0.205402,0.001988,tie,AG
8,qc__qc_decision,0.761386,0.764356,-0.002970,0.489764,0.499125,0.009360,KMeans,AG
9,paper_domain_architecture,0.789109,0.792079,-0.002970,0.388085,0.396611,0.008526,KMeans,AG


In [6]:
# =============================================================================
# CELL 6 - KMeans clusters subdivided by AG
# =============================================================================

display(kmeans_ga_matrix)
display(kmeans_subdivision)


,kmeans_cluster_label,0,1,10,11,2,3,4,5,6,7,8,9
0,0,0,0,398,0,0,0,0,2,3,0,0,0
1,1,0,0,0,0,0,0,0,0,0,67,0,0
2,2,79,0,0,0,0,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,40,0,0,0,0
4,4,0,0,0,0,0,24,0,0,0,0,0,0
5,5,0,0,4,16,0,0,0,1,200,0,0,18
6,6,0,0,0,0,0,0,39,0,0,0,0,0
7,7,0,0,0,0,0,0,0,0,0,0,66,0
8,8,0,0,0,0,13,0,0,0,0,0,0,0
9,9,0,40,0,0,0,0,0,0,0,0,0,0


,kmeans_cluster_label,kmeans_cluster_size,ga_subcluster_count,ga_subcluster_count_ge_minimum,dominant_ga_cluster_label,dominant_ga_cluster_count,dominant_ga_cluster_fraction,ga_distribution_entropy_bits,ga_distribution_normalized_entropy,top_ga_subclusters
0,5,239,5,3,6,200,0.836820,0.889026,0.382883,6:200 (83.7%); 9:18 (7.5%); 11:16 (6.7%); 10:4...
1,0,403,3,1,10,398,0.987593,0.108404,0.068395,10:398 (98.8%); 6:3 (0.7%); 5:2 (0.5%)
2,2,79,1,1,0,79,1.000000,-0.000000,0.000000,0:79 (100.0%)
3,1,67,1,1,7,67,1.000000,-0.000000,0.000000,7:67 (100.0%)
4,7,66,1,1,8,66,1.000000,-0.000000,0.000000,8:66 (100.0%)
5,3,40,1,1,5,40,1.000000,-0.000000,0.000000,5:40 (100.0%)
6,9,40,1,1,1,40,1.000000,-0.000000,0.000000,1:40 (100.0%)
7,6,39,1,1,4,39,1.000000,-0.000000,0.000000,4:39 (100.0%)
8,4,24,1,1,3,24,1.000000,-0.000000,0.000000,3:24 (100.0%)
9,8,13,1,1,2,13,1.000000,-0.000000,0.000000,2:13 (100.0%)


In [7]:
# =============================================================================
# CELL 7 - Markdown summary and HTML report
# =============================================================================

summary_text = (latest_directory / SUMMARY_FILE_NAME).read_text(encoding="utf-8")
display(Markdown(summary_text))

html_report_path = latest_directory / PLOT_HTML_FILE_NAME
display(
    HTML(
        f'<p><a href="{html_report_path.as_posix()}" target="_blank">'
        "Open AG vs KMeans comparison HTML report"
        "</a></p>"
    )
)
print(html_report_path)


# AG vs KMeans cluster comparison

This report compares the genetic algorithm (AG) clustering against the KMeans baseline using geometry, biological enrichment, homogeneity metrics, and the KMeans-to-AG subdivision matrix.

Statistical caution: protein sequences are evolutionarily related, so observations are not fully independent. Significance values should be interpreted cautiously; conclusions prioritize effect sizes, homogeneity metrics, and coherent biological subdivisions.

## Geometric and agreement metrics

- ga_cluster_label: clusters=12, silhouette=0.419, Davies-Bouldin=0.651, Calinski-Harabasz=416.0, largest cluster=39.8%
- kmeans_cluster_label: clusters=10, silhouette=0.390, Davies-Bouldin=0.808, Calinski-Harabasz=289.7, largest cluster=39.9%
- AG vs KMeans: ARI=0.930, NMI=0.939

## Biological enrichment: Cramer's V

- paper_has_piwi_catalytic_tetrad: AG=0.458, KMeans=0.382, delta=+0.077, better=AG
- paper_paz_type: AG=0.592, KMeans=0.552, delta=+0.040, better=AG
- paper_ago_type_family: AG=0.501, KMeans=0.470, delta=+0.031, better=AG
- paper_ago_type_raw: AG=0.364, KMeans=0.338, delta=+0.025, better=AG
- paper_mid_5oh_type: AG=0.189, KMeans=0.170, delta=+0.019, better=AG
- qc__primary_label: AG=0.147, KMeans=0.142, delta=+0.005, better=AG
- qc__qc_decision: AG=0.476, KMeans=0.472, delta=+0.005, better=AG
- taxonomy__04: AG=0.469, KMeans=0.465, delta=+0.004, better=AG

## Homogeneity: purity and entropy

- taxonomy__04: weighted purity AG=0.410, KMeans=0.387, delta=+0.022; normalized entropy delta(KMeans-AG)=+0.019
- taxonomy__03: weighted purity AG=0.634, KMeans=0.614, delta=+0.020; normalized entropy delta(KMeans-AG)=+0.016
- paper_ago_type_family: weighted purity AG=0.734, KMeans=0.717, delta=+0.017; normalized entropy delta(KMeans-AG)=+0.022
- paper_ago_type_raw: weighted purity AG=0.714, KMeans=0.697, delta=+0.017; normalized entropy delta(KMeans-AG)=+0.018
- paper_has_piwi_catalytic_tetrad: weighted purity AG=0.841, KMeans=0.825, delta=+0.016; normalized entropy delta(KMeans-AG)=+0.037
- paper_paz_type: weighted purity AG=0.727, KMeans=0.713, delta=+0.014; normalized entropy delta(KMeans-AG)=+0.027
- paper_mid_5oh_type: weighted purity AG=0.991, KMeans=0.991, delta=+0.000; normalized entropy delta(KMeans-AG)=+0.002
- qc__primary_label: weighted purity AG=0.963, KMeans=0.963, delta=+0.000; normalized entropy delta(KMeans-AG)=+0.002

## KMeans clusters subdivided by AG

- KMeans 5 (n=239): 3 AG subclusters >= minimum size; dominant AG=6 (83.7%); top=6:200 (83.7%); 9:18 (7.5%); 11:16 (6.7%); 10:4 (1.7%); 5:1 (0.4%)
- KMeans 0 (n=403): 1 AG subclusters >= minimum size; dominant AG=10 (98.8%); top=10:398 (98.8%); 6:3 (0.7%); 5:2 (0.5%)
- KMeans 2 (n=79): 1 AG subclusters >= minimum size; dominant AG=0 (100.0%); top=0:79 (100.0%)
- KMeans 1 (n=67): 1 AG subclusters >= minimum size; dominant AG=7 (100.0%); top=7:67 (100.0%)
- KMeans 7 (n=66): 1 AG subclusters >= minimum size; dominant AG=8 (100.0%); top=8:66 (100.0%)
- KMeans 3 (n=40): 1 AG subclusters >= minimum size; dominant AG=5 (100.0%); top=5:40 (100.0%)
- KMeans 9 (n=40): 1 AG subclusters >= minimum size; dominant AG=1 (100.0%); top=1:40 (100.0%)
- KMeans 6 (n=39): 1 AG subclusters >= minimum size; dominant AG=4 (100.0%); top=4:39 (100.0%)

## Conclusion

- AG has higher Cramer's V for 11 variables; KMeans is higher for 1.
- AG has higher weighted majority purity for 6 variables; KMeans is higher for 4.
- The main question for the final report is not whether AG is globally different from KMeans, because ARI is high; it is whether the extra AG clusters split KMeans groups into biologically coherent subgroups.


C:\Programming\Python\pAgo-project\data\04-analysis\cluster_comparison\latest\ag_vs_kmeans_cluster_comparison.html
